# A3：長任務異步 Agent — Pub/Sub 解耦 + Checkpoint 斷點續傳

> **對應 Blog：** FDE 面試準備指南（二十一）長任務 Agent 的異步分散式架構  
> **核心目標：** 用 Python 模擬 Cloud Pub/Sub + Firestore，親手實作 Checkpoint 斷點恢復，讓 Worker 崩潰不從頭來過。

## 面試情境

> 「客戶想打造一個自動化市場競品分析 Agent。整個工作流需要 30~60 分鐘。你如何設計後端分散式架構？如果執行到第 25 分鐘時某個節點崩潰，如何確保不從頭來過？」

## 學習路徑

| Part | 內容 | 模擬的生產組件 |
|------|------|---------------|
| 1 | 為什麼同步 HTTP 不行 | 展示超時問題 |
| 2 | 任務佇列（In-memory Pub/Sub） | Cloud Pub/Sub |
| 3 | Checkpoint 設計 | Redis + Firestore |
| 4 | Worker 崩潰與恢復 | GKE Pod 重啟 |
| 5 | 進度回報（Polling + SSE） | 前端進度顯示 |
| 6 | 面試白板語言 | |

In [ ]:
import asyncio
import time
import uuid
import json
import copy
import random
from enum import Enum
from typing import Any, Optional
from dataclasses import dataclass, field

print("環境就緒")

## Part 1：為什麼同步 HTTP 不行

In [ ]:
# 展示同步模型的三個致命限制

async def simulate_sync_problem():
    """模擬：用戶必須等 HTTP Response 的問題"""
    
    HTTP_TIMEOUT = 5.0  # 模擬 Load Balancer timeout = 5 秒（生產通常 30~300s）
    AGENT_RUNTIME = 8.0  # Agent 需要 8 秒（生產通常 30~60 分鐘）
    
    print(f"模擬場景：")
    print(f"  HTTP Timeout: {HTTP_TIMEOUT}s")
    print(f"  Agent 執行時間: {AGENT_RUNTIME}s")
    print()
    
    # 模擬用戶等待
    start = time.time()
    print("用戶發出請求...")
    
    try:
        # 模擬「必須在 timeout 內完成」的限制
        await asyncio.wait_for(
            asyncio.sleep(AGENT_RUNTIME),  # Agent 在跑
            timeout=HTTP_TIMEOUT
        )
        print("回應成功")
    except asyncio.TimeoutError:
        elapsed = time.time() - start
        print(f"\n💥 [{elapsed:.1f}s] TIMEOUT！用戶看到 504 Gateway Timeout")
        print("   Agent 可能還在跑，但用戶已斷線")
        print("   Agent 的進度全部消失（沒有 Checkpoint）")
        print("   用戶必須從頭再來")

print("=" * 50)
print("❌ 同步 HTTP 模型的問題")
print("=" * 50)
await simulate_sync_problem()

print("""
同步模型的三個致命限制：
  1. HTTP timeout: Load Balancer 強制斷線（30~300s）
  2. 無法容錯: Worker 崩潰 → 用戶從頭來過
  3. 資源浪費: 一個請求佔用一個 Thread 60 分鐘
""")

## Part 2：解耦架構 — In-memory 模擬 Cloud Pub/Sub

In [ ]:
# ✅ 模擬 Cloud Pub/Sub（用 asyncio.Queue）

class TaskStatus(Enum):
    PENDING = "pending"
    RUNNING = "running"
    COMPLETED = "completed"
    FAILED = "failed"
    TIMEOUT = "timeout"

@dataclass
class Task:
    task_id: str
    payload: dict
    status: TaskStatus = TaskStatus.PENDING
    progress: int = 0
    result: Any = None
    error: str = None
    created_at: float = field(default_factory=time.time)

# 模擬系統組件
task_queue: asyncio.Queue = asyncio.Queue()     # 模擬 Cloud Pub/Sub
task_store: dict[str, Task] = {}                # 模擬 Firestore（任務狀態）
checkpoint_store: dict[str, dict] = {}          # 模擬 Redis（Checkpoint）


# API Gateway：接收請求，立即回傳 task_id（< 100ms）
async def submit_task(payload: dict) -> str:
    """模擬 POST /tasks — 立即回傳，不等 Agent 完成"""
    task_id = str(uuid.uuid4())[:8]
    task = Task(task_id=task_id, payload=payload)
    
    # 儲存到 Firestore
    task_store[task_id] = task
    
    # 寫入 Pub/Sub Queue
    await task_queue.put(task_id)
    
    print(f"[API] 任務已提交，task_id={task_id}（< 100ms 回應）")
    return task_id


# 查詢任務狀態
def get_task_status(task_id: str) -> dict:
    """模擬 GET /tasks/{task_id}/status"""
    task = task_store.get(task_id)
    if not task:
        return {"error": "Task not found"}
    return {
        "task_id": task.task_id,
        "status": task.status.value,
        "progress": task.progress,
        "result": task.result
    }

print("✅ 解耦架構組件初始化完成")
print("   task_queue: 模擬 Cloud Pub/Sub")
print("   task_store: 模擬 Firestore")
print("   checkpoint_store: 模擬 Redis")

## Part 3：Checkpoint 設計 — 每個步驟完成後存檔

In [ ]:
# ✅ Checkpoint 工具函數

def save_checkpoint(task_id: str, step_name: str, step_result: Any):
    """模擬寫入 Redis（Hot Path）+ Firestore（持久化）"""
    if task_id not in checkpoint_store:
        checkpoint_store[task_id] = {"steps": {}}
    
    checkpoint_store[task_id]["steps"][step_name] = {
        "result": step_result,
        "saved_at": time.time()
    }
    print(f"  💾 [Checkpoint] 步驟 '{step_name}' 已儲存")

def load_checkpoint(task_id: str) -> dict:
    """讀取最後的 Checkpoint（模擬從 Redis/Firestore 讀）"""
    return checkpoint_store.get(task_id, {}).get("steps", {})

def is_step_done(task_id: str, step_name: str) -> bool:
    """判斷某個步驟是否已完成（避免重複執行）"""
    steps = load_checkpoint(task_id)
    return step_name in steps

def get_step_result(task_id: str, step_name: str) -> Any:
    """取得已完成步驟的結果"""
    steps = load_checkpoint(task_id)
    return steps.get(step_name, {}).get("result")

print("✅ Checkpoint 工具函數設計完成")

In [ ]:
# 定義 Agent 的工作步驟（模擬市場競品分析）

STEPS = [
    ("爬取競品網站",      1.0),  # 步驟名稱, 模擬執行時間（縮短為 1~2s）
    ("提取關鍵資訊",     0.8),
    ("比較功能矩陣",     0.7),
    ("生成分析初稿",      1.2),
    ("撰寫最終報告",     0.9),
]

async def execute_step(task_id: str, step_name: str, duration: float, 
                        crash_at: str = None) -> Any:
    """執行單一步驟，並在完成後儲存 Checkpoint"""
    
    # 如果已有 Checkpoint，跳過（斷點恢復）
    if is_step_done(task_id, step_name):
        result = get_step_result(task_id, step_name)
        print(f"  ⏭️  [跳過] '{step_name}'（已有 Checkpoint）")
        return result
    
    # 執行步驟
    print(f"  ▶️  [執行] '{step_name}'...")
    
    # 模擬 Worker 崩潰
    if crash_at == step_name:
        await asyncio.sleep(duration * 0.3)  # 跑到一半就崩潰
        raise RuntimeError(f"Worker 崩潰於步驟：{step_name}")
    
    await asyncio.sleep(duration)  # 模擬執行時間
    
    result = {"status": "done", "step": step_name, "tokens_used": random.randint(1000, 5000)}
    
    # 儲存 Checkpoint
    save_checkpoint(task_id, step_name, result)
    
    return result

print("工作步驟定義完成：")
for name, duration in STEPS:
    print(f"  {name}: {duration}s")

## Part 4：Worker 崩潰與恢復（核心實作）

In [ ]:
# ✅ Agent Worker（可模擬崩潰）

async def agent_worker(task_id: str, crash_at: str = None):
    """Worker 執行 Agent 任務，支援 Checkpoint 斷點恢復"""
    
    task = task_store[task_id]
    task.status = TaskStatus.RUNNING
    
    # 使用 Redis 分散式 Lock（模擬）避免重複執行
    print(f"\n[Worker] 開始執行 task_id={task_id}")
    if crash_at:
        print(f"[Worker] ⚠️  將在步驟 '{crash_at}' 崩潰")
    
    total_tokens = 0
    
    try:
        for i, (step_name, duration) in enumerate(STEPS):
            result = await execute_step(task_id, step_name, duration, crash_at)
            total_tokens += result.get("tokens_used", 0)
            
            # 更新進度（前端 Polling 會看到這個）
            task.progress = int((i + 1) / len(STEPS) * 100)
        
        # 任務完成
        task.status = TaskStatus.COMPLETED
        task.result = {
            "report": "競品分析報告已完成",
            "total_tokens": total_tokens
        }
        print(f"\n✅ 任務完成！總 Token 消耗: {total_tokens}")
        
    except RuntimeError as e:
        task.status = TaskStatus.FAILED
        task.error = str(e)
        print(f"\n💥 [{str(e)}]")
        raise


print("Worker 函數設計完成")

In [ ]:
# 🎬 完整的崩潰 + 恢復演示

CRASH_STEP = "比較功能矩陣"  # 在這個步驟模擬崩潰

print("=" * 55)
print("🎬 完整演示：Worker 崩潰 → Checkpoint 恢復")
print("=" * 55)

# Step 1: 用戶提交任務
task_id = await submit_task({"target": "競品分析", "companies": ["A公司", "B公司", "C公司"]})
print()

# Step 2: Worker 1 執行（在指定步驟崩潰）
print("=" * 55)
print("Worker 1 執行（模擬崩潰）")
print("=" * 55)
try:
    await agent_worker(task_id, crash_at=CRASH_STEP)
except RuntimeError:
    pass  # Worker 崩潰

print(f"\n已完成的 Checkpoint：")
completed_steps = list(load_checkpoint(task_id).keys())
for s in completed_steps:
    print(f"  ✓ {s}")
print(f"  ✗ {CRASH_STEP}（崩潰，未儲存）")

# Step 3: 排程器偵測到 Worker 崩潰，重新分配任務
print()
print("=" * 55)
print("排程器偵測心跳停止 → 重新分配任務給 Worker 2")
print("=" * 55)

# 重置任務狀態（讓 Worker 2 可以接手）
task_store[task_id].status = TaskStatus.PENDING

# Worker 2 從 Checkpoint 恢復（不重新執行已完成的步驟）
await agent_worker(task_id, crash_at=None)  # 這次不崩潰

# 查看最終狀態
final_status = get_task_status(task_id)
print(f"\n📊 最終任務狀態：")
print(f"  status: {final_status['status']}")
print(f"  progress: {final_status['progress']}%")
print(f"  result: {final_status['result']}")

## Part 5：進度回報設計（Polling vs SSE）

In [ ]:
# ✅ 模擬前端 Polling 進度

async def frontend_poller(task_id: str, interval: float = 0.5, max_polls: int = 30):
    """模擬前端每隔 N 秒問一次進度"""
    print(f"\n[前端] 開始 Polling 進度（每 {interval}s 詢問一次）")
    
    for i in range(max_polls):
        await asyncio.sleep(interval)
        status = get_task_status(task_id)
        print(f"  [{i+1:2d}] status={status['status']:10s} progress={status['progress']:3d}%")
        
        if status["status"] in ["completed", "failed", "timeout"]:
            print(f"\n[前端] 任務結束：{status['status']}")
            if status.get("result"):
                print(f"[前端] 結果: {status['result']}")
            return
    
    print("[前端] Polling 超時")


# 重設環境，做一次完整的 end-to-end 演示
task_store.clear()
checkpoint_store.clear()
while not task_queue.empty():
    task_queue.get_nowait()

print("=" * 55)
print("完整 E2E 演示：提交任務 + Polling 進度")
print("=" * 55)

# 同時執行：Worker 跑任務 + 前端 Polling
new_task_id = await submit_task({"target": "快速分析"})
print()

await asyncio.gather(
    agent_worker(new_task_id),
    frontend_poller(new_task_id, interval=0.3, max_polls=30)
)

## Part 6：面試白板語言 + Trade-off

In [ ]:
print("""
面試答題框架：
─────────────────────────────────────────────────────
長任務不能用同步 HTTP 模型，因為 30~60 分鐘遠超過
任何 Load Balancer 的 timeout，且無法容錯。

我的設計是解耦架構：
  ① API Gateway 收到請求後立即寫入 Cloud Pub/Sub，
    並回傳 task_id（< 100ms）
  ② GKE 上的 Worker Pool 異步消費任務，
    可以根據 Queue 深度自動水平擴縮

容錯靠 Checkpoint：
  每完成一個有意義的步驟，就把當前狀態寫入：
  ├── Redis：熱狀態（< 1ms 讀寫，Worker 崩潰後快速恢復）
  └── Firestore：完整歷史（持久化，支援查詢）
  Worker 崩潰後，排程器重新分配任務，
  新 Worker 從最後的 Checkpoint 繼續，不重複消耗 Token。

進度回報：
  ├── Polling：簡單，前後端容易，適合大多數場景
  ├── SSE：即時，單向推送，輕量
  └── WebSocket：需要雙向互動（如 Human-in-the-loop 確認）

Edge Cases：
  ├── 同一任務被多個 Worker 同時領取 → Redis Distributed Lock
  ├── 任務永遠不結束 → 硬性 TTL（2 小時）
  └── 429 限流 → Exponential Backoff + 任務重排
─────────────────────────────────────────────────────
""")

print("Trade-off 速查：")
print("""
Q: Checkpoint 粒度要多細？
   太粗（每個大階段）: 崩潰後重做太多工作
   太細（每個 LLM 呼叫）: Firestore 寫入成本高、速度慢
   推薦: 每個「有意義的獨立子任務」完成後存一次

Q: 為什麼用 Pub/Sub 而不是直接丟給 Worker？
   Pub/Sub: 持久化佇列，Worker 崩潰後消息不丟失，at-least-once delivery
   直接呼叫: 簡單，但 Worker 崩潰 = 任務丟失
   選 Pub/Sub 的關鍵: 任務必須能容錯

Q: 結果太大（20 頁報告）怎麼回傳？
   報告存 GCS，Firestore 只存 URI，用戶透過 Signed URL 下載
   避免: 把大型結果直接塞進 Firestore 或 HTTP Response
""")